# Memory 记忆管理教程 (v0.5)

本教程帮助你理解 Memory 长期记忆的概念和使用方式。

## 学习目标

1. 理解短期记忆 vs 长期记忆
2. 掌握记忆的添加、检索、遗忘操作
3. 学会记忆分类和重要性管理
4. 理解 Memory 与 Session 的集成

In [1]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

依赖安装完成！


In [2]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

项目根目录: d:\SVN\RogueRabbit
源码路径: d:\SVN\RogueRabbit\src


## 1. 什么是 Memory？

Memory 是**长期知识存储**，用于跨会话保存和检索信息。

### Session vs Memory

| 特性 | Session | Memory |
|------|---------|--------|
| 类型 | 短期对话历史 | 长期知识存储 |
| 生命周期 | 随会话关闭 | 跨会话持久 |
| 内容 | 完整对话消息 | 提取的关键信息 |
| 检索 | 按顺序回放 | 按关键词/分类搜索 |

In [3]:
# 导入 Memory 相关类型
from rogue_rabbit.contracts import (
    Memory, MemoryItem, MemoryMeta,
    Message, Role
)
from rogue_rabbit.core import MemoryManager
from rogue_rabbit.runtime import InMemoryStore

# 创建记忆管理器
manager = MemoryManager(store=InMemoryStore())
print("MemoryManager 创建成功")

MemoryManager 创建成功


## 2. 记忆生命周期

记忆空间 -> 添加记忆 -> 检索记忆 -> 遗忘记忆

In [4]:
# 创建记忆空间
memory = manager.create(user_id="demo_user")
print(f"记忆空间ID: {memory.meta.memory_id}")
print(f"用户: {memory.meta.user_id}")

记忆空间ID: 0c5dcb88
用户: demo_user


In [5]:
# 添加记忆
manager.add_memory("demo_user", "用户的名字叫小明", importance=0.9, category="fact")
manager.add_memory("demo_user", "用户喜欢Python", importance=0.8, category="preference")
manager.add_memory("demo_user", "用户是学生", importance=0.7, category="fact")

# 查看
memory = manager.get("demo_user")
print(f"共 {len(memory.items)} 条记忆:")
for item in memory.items:
    print(f"  [{item.category}] {item.content} (重要性: {item.importance})")

共 3 条记忆:
  [fact] 用户的名字叫小明 (重要性: 0.9)
  [preference] 用户喜欢Python (重要性: 0.8)
  [fact] 用户是学生 (重要性: 0.7)


In [6]:
# 搜索记忆
results = manager.search("demo_user", "Python")
print(f"搜索 'Python': {len(results)} 条结果")
for r in results:
    print(f"  - {r.content}")

搜索 'Python': 1 条结果
  - 用户喜欢Python


In [7]:
# 遗忘记忆
removed = manager.forget("demo_user", "小明")
print(f"遗忘包含'小明'的记忆: {removed} 条")

memory = manager.get("demo_user")
print(f"剩余 {len(memory.items)} 条记忆")

遗忘包含'小明'的记忆: 1 条
剩余 2 条记忆


## 3. 分类和重要性

记忆支持分类标签和重要性评分：
- **分类**: fact, preference, skill, event 等
- **重要性**: 0.0-1.0，越高越重要

In [8]:
# 按分类获取
facts = memory.get_by_category("preference")
print(f"preference 分类: {len(facts)} 条")
for item in facts:
    print(f"  - {item.content}")

# 按重要性获取
important = memory.get_important(threshold=0.7)
print(f"\n重要性 >= 0.7: {len(important)} 条")
for item in important:
    print(f"  - [{item.category}] {item.content}")

preference 分类: 1 条
  - 用户喜欢Python

重要性 >= 0.7: 2 条
  - [preference] 用户喜欢Python
  - [fact] 用户是学生


## 4. 与 Session 集成

Memory 的核心价值在于与 Session 配合使用：
1. 从对话中提取关键信息存入 Memory
2. 新对话时检索相关记忆注入上下文

In [9]:
# 预存记忆
manager2 = MemoryManager(store=InMemoryStore())
manager2.create(user_id="alice")
manager2.add_memory("alice", "用户喜欢简洁回答", importance=0.8, category="preference")
manager2.add_memory("alice", "用户是数据分析师", importance=0.9, category="fact")

# 获取记忆上下文
context = manager2.get_context_for_session("alice", "数据分析")
print("注入到 Session 的记忆:")
print(context)

注入到 Session 的记忆:
[相关记忆]
- [fact] 用户是数据分析师


## 总结

### 核心概念

1. **MemoryItem**: 单条记忆（内容、重要性、分类）
2. **Memory**: 记忆空间（一个用户的所有长期记忆）
3. **MemoryManager**: 记忆管理器

### Session vs Memory

- Session: 短期对话历史
- Memory: 长期知识存储
- 集成: 记忆注入对话上下文

### 下一步

- 运行 `experiments/12_memory_basic.py`
- 运行 `experiments/13_memory_with_session.py`
- 尝试 FileMemoryStore 持久化